# Energy Dataset: Interactive Scientific Audit
**Interactive Dashboard**

This notebook provides a complete interactive tool to audit energy datasets (RTE vs Enedis),
check physical consistency, and visualize weather correlations.

**Features:**
* **Dynamic File Loading**: Scans the `Outputs/` directory for available datasets.
* **Smart Plotting**: Automatic double-axis handling for multi-variate analysis.
* **Anomaly Detection**: Automatic "Cold Spell" detection and zooming.
* **Structural Audit**: Visualizes data availability (gaps) via heatmaps.

In [ ]:
# SPDX-FileCopyrightText: 2025-2026 EDF (Electricité De France)
# SPDX-License-Identifier: LGPL-3.0-or-later
# Author: Yann Allioux

In [ ]:
import ast
import glob
import math
import os
import re
from typing import Dict, List, Optional

import ipywidgets as widgets
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import clear_output, display

# Graphical configuration
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.figsize'] = (15, 6)

# File and features configuration
OUTPUT_DIR = "Outputs"
TEMP_COL_REF = "meteo_temperature_celsius_france_load"

print("✅ Environment loaded.")

## 1. Data Loading
Select a dataset from the `Outputs` directory using the dropdown below.
The system automatically groups `15min` and `30min` versions of the same dataset.

In [ ]:
# Global state management for the notebook
if 'df_15' not in globals(): df_15 = None
if 'df_30' not in globals(): df_30 = None
if 'df_main' not in globals(): df_main = None

def scan_and_group_datasets() -> Dict[str, Dict[str, str]]:
    """Scans the output directory and groups files by dataset name."""
    # Safety check for directory existence
    if not os.path.exists(OUTPUT_DIR):
        print(f"⚠️ Directory '{OUTPUT_DIR}' not found. Please run the pipeline first.")
        return {}

    files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.csv"), recursive=True))
    dataset_map = {}
    
    pattern = re.compile(r"dataset_(15|30)min_(.+)\.csv")
    
    for file_path in files:
        filename = os.path.basename(file_path)
        match = pattern.search(filename)
        
        if match:
            resolution = match.group(1) # '15' or '30'
            name = match.group(2)       # e.g., 'full_imputed'
            
            if name not in dataset_map:
                dataset_map[name] = {'15': None, '30': None}
            
            dataset_map[name][resolution] = file_path
            
    return dataset_map

datasets_dict = scan_and_group_datasets()
dataset_names = sorted(datasets_dict.keys()) if datasets_dict else ["No Data Found"]

w_dataset_select = widgets.Dropdown(
    options=dataset_names,
    description='Dataset:',
    layout={'width': '50%'}
)

# Set default selection if available
for name in ['full_imputed', 'full', 'standard_imputed']:
    if name in dataset_names:
        w_dataset_select.value = name
        break

btn_load = widgets.Button(description='Load Dataset', button_style='success', icon='upload')
out_load = widgets.Output()

def load_data(b):
    # UI Feedback
    original_desc = b.description
    b.disabled = True
    b.description = "Loading..."
    out_load.clear_output()
    
    global df_15, df_30, df_main
    df_15, df_30, df_main = None, None, None
    
    selected_name = w_dataset_select.value
    if selected_name == "No Data Found":
        with out_load: print("❌ No datasets available.")
        b.disabled = False
        b.description = original_desc
        return

    files_to_load = datasets_dict[selected_name]
    
    with out_load:
        try:
            print(f"⏳ Loading dataset collection: '{selected_name}'...")
            
            # Load 15min version
            path_15 = files_to_load.get('15')
            if path_15:
                df_15 = pd.read_csv(path_15, sep=';', parse_dates=['date'])
                if 'date' in df_15.columns: df_15 = df_15.sort_values('date').set_index('date')
                print(f"   ✅ 15 min loaded ({len(df_15)} rows)")
            
            # Load 30min version
            path_30 = files_to_load.get('30')
            if path_30:
                df_30 = pd.read_csv(path_30, sep=';', parse_dates=['date'])
                if 'date' in df_30.columns: df_30 = df_30.sort_values('date').set_index('date')
                print(f"   ✅ 30 min loaded ({len(df_30)} rows)")

            # Define Main DataFrame (Prefer 30min for speed in charts, fallback to 15min)
            df_main = df_30 if df_30 is not None else df_15
            
            if df_main is not None:
                print("-" * 40)
                print(f"🚀 READY: Active Main Dataset set to {'30 min' if df_main is df_30 else '15 min'}.")
            else:
                print("❌ ERROR: Files exist but could not be loaded.")

        except Exception as e:
            print(f"❌ FATAL ERROR: {e}")
            import traceback
            traceback.print_exc()
        finally:
            b.disabled = False
            b.description = 'Load Dataset'

# This line is crucial in Jupyter environments where cells are re-run.
btn_load._click_handlers.callbacks = [] 
btn_load.on_click(load_data)

display(widgets.VBox([
    widgets.HTML("<h3>📂 Smart Data Loader</h3>"),
    widgets.HBox([w_dataset_select, btn_load]),
    out_load
]))

## 2. Hierarchy Definitions & Metadata
Defining physical relationships and groupings for automated plotting.

In [13]:
REGIONS_SLUGS = [
    'auvergne_rhone_alpes', 'bourgogne_franche_comte', 'bretagne', 
    'centre_val_de_loire', 'grand_est', 'hauts_de_france', 'ile_de_france', 
    'normandie', 'nouvelle_aquitaine', 'occitanie', 'pays_de_la_loire', 
    'provence_alpes_cote_d_azur', 'corse'
]

# Physical Balance Checks (Conservation of Energy)
ENEDIS_BALANCE_CHECKS = [
    {
        "In": ["enedis_production_total_france", "enedis_net_import_from_rte_france"],
        "Out": ["enedis_load_total_france", "enedis_net_export_to_rte_france", "enedis_grid_losses_france",
            "enedis_net_export_to_dso_france"],
        "Title": "Physical Balance: Enedis Global (In = Out)"
    },
    {
        "In": ["enedis_load_total_france"],
        "Out": [
            "enedis_load_industrial_total_france", 
            "enedis_load_residential_total_france", 
            "enedis_load_professional_total_france", 
            "enedis_load_entreprise_total_france"
        ],
        "Title": "Load Consistency (Sectors + Losses)"
    },
    {
        "In": ["enedis_production_total_france"],
        "Out": [
            "enedis_production_solar_total_france", 
            "enedis_production_wind_total_france", 
            "enedis_production_hydro_total_france", 
            "enedis_production_bioenergy_total_france", 
            "enedis_production_other_total_france"
        ],
        "Title": "Production Consistency (Sum of Technologies)"
    }
]

def get_regional_groups(columns):
    """Auto-detects regional columns and groups them by physical variable."""
    groups = []
    
    definitions = [
        # 1. Meteo Specific (Weighted)
        ("Regional: Temperature (Load Weighted)", ["meteo_temperature_celsius_", "_load"]),
        ("Regional: Temperature (Solar Weighted)", ["meteo_temperature_celsius_", "_solar"]),
        ("Regional: Nebulosity (Load Weighted)", ["meteo_nebulosity_", "_load"]),
        ("Regional: Nebulosity (Solar Weighted)", ["meteo_nebulosity_", "_solar"]),
        ("Regional: Wind Speed (Load Weighted)", ["meteo_wind_speed_ms_", "_load"]),
        ("Regional: Wind Speed (Wind Weighted)", ["meteo_wind_speed_ms_", "_wind"]),
        ("Regional: Air Density (Wind Weighted)", ["meteo_air_density_kgm3_", "_wind"]),
        ("Regional: Humidity (Load Weighted)", ["meteo_humidity_percent_", "_load"]), 
        ("Regional: Wind Direction (Wind Weighted)", ["meteo_wind_direction_deg_", "_wind"]),
        ("Regional: Wind Gusts (Wind Weighted)", ["meteo_wind_gust_10min_ms_", "_wind"]),
        ("Regional: Dew Point", ["meteo_dew_point_", "_load"]),
        ("Regional: Precipitation", ["meteo_precip_", "_load"]),
        ("Regional: Pressure", ["meteo_pressure_", "_load"]),

        # 2. Production & Load (RTE)
        ("Regional: Load (RTE)", ["rte_load_"]),
        ("Regional: Solar Prod (RTE)", ["rte_solar_"]),
        ("Regional: Wind Prod (RTE)", ["rte_wind_"]),
        ("Regional: Nuclear Prod", ["rte_nuclear_"]),
        ("Regional: Hydro Prod", ["rte_hydro_"]),
        ("Regional: Bioenergy Prod", ["rte_bioenergy_"]),
        ("Regional: Pumping", ["rte_pump_"]),
        ("Regional: Physical Exchange", ["rte_exchange_net_"]),
        
        # 3. Generic Fallbacks
        ("Regional: Meteo Temperature (Other)", ["meteo_temperature_"]),
        ("Regional: Meteo Wind (Other)", ["meteo_wind_"])
    ]
    
    used_cols = set()

    for title, patterns in definitions:
        matched = []
        for col in columns:
            if col in used_cols: continue
            
            has_region = any(r in col for r in REGIONS_SLUGS)
            has_patterns = all(p in col for p in patterns)
            
            if has_region and has_patterns:
                matched.append(col)
                
        if matched:
            groups.append({"type": "region_grid", "cols": sorted(matched), "title": title})
            used_cols.update(matched)
            
    return groups

def get_calendar_group(columns):
    cal_keywords = ['day_type', 'period_', 'holiday', 'week_number', 'month', 'year', 'toy', 'tod', 'validity']
    return [c for c in columns if any(k in c.lower() for k in cal_keywords)]

def get_balance_check(columns):
    valid_checks = []
    for check in ENEDIS_BALANCE_CHECKS:
        needed = check["In"] + check["Out"]
        if all(c in columns for c in needed):
            valid_checks.append(check)
    return valid_checks

## 3. Smart Plotting Engine
Core visualization logic for Time Series, Heatmaps, and Regional Grids.

In [14]:
def find_france_ref(col_name, all_cols):
    """
    Heuristic: Tries to find the corresponding 'National' column for a given regional column.
    Used to plot the National average/sum as a reference line on regional charts.
    """
    rules = {
        'rte_load_': 'rte_load_france',
        'rte_solar_': 'rte_solar_france',
        'rte_wind_': 'rte_wind_france',
        'rte_nuclear_': 'rte_nuclear_france',
        'rte_hydro_': 'rte_hydro_france',
        'rte_bioenergy_': 'rte_bioenergy_france',
        'rte_pump_': 'rte_pump_france',
        'rte_exchange_net_': 'rte_exchange_net_france',
        'enedis_load_': 'enedis_load_total_france',
        'enedis_production_': 'enedis_production_total_france',
        'meteo_temperature_': 'meteo_temperature_celsius_france',
        'meteo_nebulosity_': 'meteo_nebulosity_france',
        'meteo_wind_speed_': 'meteo_wind_speed_ms_france',
        'meteo_air_density_': 'meteo_air_density_kgm3_france'
    }

    # 1. Check prefixes
    for prefix, national_candidate in rules.items():
        if prefix in col_name:
            if 'meteo_' in prefix:
                if '_load' in col_name: national_candidate += '_load'
                elif '_solar' in col_name: national_candidate += '_solar'
                elif '_wind' in col_name: national_candidate += '_wind'
                elif '_air' in col_name: national_candidate += '_wind'           

            if national_candidate in all_cols:
                return national_candidate

    # 2. Heuristic check (replace region name with 'france')
    for reg in sorted(REGIONS_SLUGS, key=len, reverse=True):
        if reg in col_name:
            candidates = [
                col_name.replace(reg, 'france')
            ]
            for c in candidates:
                if c in all_cols: return c
                
    return None

def plot_analysis(df, config_list, date_min, date_max, mode='long_term', window_days=7):
    """
    Main loop to generate plots based on the configuration list.
    Supports: Calendar Heatmaps, Step Lines, Balance Checks, Regional Grids, and Pairwise Comparisons.
    """
    audit_data = [] 
    mask = (df.index >= date_min) & (df.index <= date_max)
    df_sub = df.loc[mask].copy()
    
    if df_sub.empty: 
        print("⚠️ No data available for this selected period.")
        return None
    
    # Helper for nice date axis formatting
    def apply_smart_xaxis(ax):
        time_delta = df_sub.index.max() - df_sub.index.min()
        
        if time_delta.days > 365:
            fmt = mdates.DateFormatter('%b %Y')
            rot = 0
        elif time_delta.days > 5:
            fmt = mdates.DateFormatter('%Y-%m-%d')
            rot = 15
        else:
            fmt = mdates.DateFormatter('%m-%d\n%H:%M')
            rot = 0
            
        ax.xaxis.set_major_formatter(fmt)
        plt.setp(ax.get_xticklabels(), rotation=rot, ha="center")
        ax.set_xlabel("")

    # Determine title prefix and handle "Cold Spell" logic
    title_pref = f"GLOBAL VIEW ({window_days}d)"
    if mode == 'zoom_cold' and TEMP_COL_REF in df_sub.columns:
        # Find the coldest rolling week (Demand Peak Proxy)
        cold_date = df_sub[TEMP_COL_REF].resample('1h').mean().rolling(24*7).mean().idxmin()
        if pd.notna(cold_date):
            s, e = cold_date - pd.Timedelta(days=window_days), cold_date + pd.Timedelta(days=window_days/2)
            df_sub = df_sub.loc[s:e]
            title_pref = f"COLD ZOOM ({cold_date.strftime('%Y-%m-%d')})"

    # Resample to 1h for plotting performance
    df_sub = df_sub.resample('1h').mean()

    for i, item in enumerate(config_list, 1):
        
        # --- Type 1: Calendar Heatmap ---
        if item.get('type') == 'calendar_heatmap':
            cols = [c for c in item['cols'] if c in df_sub.columns]
            if not cols: continue
            for c in cols: audit_data.append({"Graph #": i, "Type": "Calendar", "Variable": c})
            
            d_norm = df_sub[cols].apply(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else x)
            fig, ax = plt.subplots(figsize=(15, max(4, len(cols)*0.4)))
            sns.heatmap(d_norm.T, ax=ax, cmap="YlGnBu", cbar=False, xticklabels=False)
            ax.set_title(f"Graph #{i} : {item['title']}")
            plt.tight_layout(); plt.show()
            continue

        # --- Type 2: Calendar Step Lines (Normalized) ---
        if item.get('type') == 'calendar_lines':
            cols = [c for c in item['cols'] if c in df_sub.columns]
            if not cols: continue
            for c in cols: audit_data.append({"Graph #": i, "Type": "Calendar", "Variable": c})
            
            fig, ax = plt.subplots(figsize=(15, max(4, len(cols) * 0.8)))
            cmap = matplotlib.colormaps['tab20']
            
            yticks_pos, yticks_labels = [], []

            for idx, col in enumerate(cols):
                series = df_sub[col].dropna()
                if series.empty: continue

                vmin, vmax = series.min(), series.max()
                series_norm = (series - vmin) / (vmax - vmin) if vmax != vmin else series - vmin
                
                offset = idx * 1.1
                clean_label = col.replace('period_', '').replace('day_type_', '').replace('holiday_', '').title()
                
                ax.step(series_norm.index, series_norm + offset, where='post', color=cmap(idx % 20), lw=2)
                ax.axhline(y=offset, color='grey', alpha=0.1, lw=1)
                
                yticks_pos.append(offset + 0.5)
                yticks_labels.append(f"{clean_label}\n[{vmin:.0f}-{vmax:.0f}]")

            ax.set_yticks(yticks_pos)
            ax.set_yticklabels(yticks_labels, fontsize=9)
            ax.set_title(f"Graph #{i} : {item['title']} (Normalized)")
            ax.grid(True, axis='x', alpha=0.3)
            
            apply_smart_xaxis(ax)
            plt.tight_layout(); plt.show()
            continue

        # --- Type 3: Balance Check (In vs Out) ---
        if "In" in item:
            cols_in = [c for c in item["In"] if c in df_sub.columns]
            cols_out = [c for c in item["Out"] if c in df_sub.columns]
            
            if not cols_in or not cols_out: continue
            
            sum_in = df_sub[cols_in].sum(axis=1)
            sum_out = df_sub[cols_out].sum(axis=1)
            delta = sum_in - sum_out
            
            for c in cols_in + cols_out: audit_data.append({"Graph #": i, "Type": "Balance", "Variable": c})
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True, 
                                           gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05})

            ax1.plot(sum_in.index, sum_in, label="IN (Total)", color='green', lw=2.5, alpha=0.8)
            ax1.plot(sum_out.index, sum_out, label="OUT (Total)", color='red', lw=2, ls='--', alpha=0.9)
            
            ax1.set_title(f"Graph #{i} : {item['Title']} - {title_pref}", fontsize=14, fontweight='bold')
            ax1.legend(loc='upper right')
            ax1.grid(True, alpha=0.3)
            ax1.set_ylabel("Power (MW)")

            ax2.plot(delta.index, delta, color='black', lw=1, label="Delta (In - Out)")
            
            ax2.fill_between(delta.index, delta, 0, where=(delta > 0), color='green', alpha=0.3)
            ax2.fill_between(delta.index, delta, 0, where=(delta < 0), color='red', alpha=0.3)
            
            ax2.axhline(0, color='black', lw=1, ls='-')
            
            max_err = delta.abs().max()
            mean_err = delta.mean()
            stats_str = f"Max Err: {max_err:.2f} MW | Mean Err: {mean_err:.2f} MW"
            
            ax2.set_ylabel("Delta (MW)")
            ax2.legend(loc='upper right', title=stats_str)
            ax2.grid(True, alpha=0.3)
            
            apply_smart_xaxis(ax2)
            plt.show()
            continue

        # --- Type 4: Regional Grid ---
        if item.get('type') == 'region_grid':
            cols = [c for c in item['cols'] if c in df_sub.columns]
            if not cols: continue
            for c in cols: audit_data.append({"Graph #": i, "Type": "Regional", "Variable": c})

            nat_col = find_france_ref(cols[0], df.columns)

            # --- SUB-PLOT 1: Sum/Mean vs National WITH Delta ---
            if nat_col:
                is_meteo = 'meteo' in cols[0]
                sum_reg = df_sub[cols].mean(axis=1) if is_meteo else df_sub[cols].sum(axis=1)
                agg_label = "Mean of Regions" if is_meteo else "Sum of Regions"
                nat_val = df_sub[nat_col]
                
                # Calculate Delta
                delta = sum_reg - nat_val
                
                # Setup 2 subplots like Balance Check
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True, 
                                               gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05})

                # Top: Comparison
                ax1.plot(sum_reg.index, sum_reg, label=agg_label, color='#2ca02c', lw=2.5, alpha=0.9) # Green-ish
                ax1.plot(nat_val.index, nat_val, label=f"National ({nat_col})", color='black', ls='--', lw=2, alpha=0.8)
                
                ax1.set_title(f"Graph #{i} (Spatial Consistency) : {item['title']} - {title_pref}", fontsize=14, fontweight='bold')
                ax1.legend(loc='upper right')
                ax1.grid(True, alpha=0.3)
                ax1.set_ylabel("Value")

                # Bottom: Delta
                ax2.plot(delta.index, delta, color='black', lw=1, label=f"Delta ({agg_label} - National)")
                
                # Green if Agg > National, Red if Agg < National
                ax2.fill_between(delta.index, delta, 0, where=(delta > 0), color='green', alpha=0.3)
                ax2.fill_between(delta.index, delta, 0, where=(delta < 0), color='red', alpha=0.3)
                
                ax2.axhline(0, color='black', lw=1, ls='-')
                
                max_err = delta.abs().max()
                mean_err = delta.mean()
                stats_str = f"Max Diff: {max_err:.2f} | Mean Diff: {mean_err:.2f}"
                
                ax2.set_ylabel("Delta")
                ax2.legend(loc='upper right', title=stats_str)
                ax2.grid(True, alpha=0.3)
                
                apply_smart_xaxis(ax2)
                plt.show()

            # --- SUB-PLOT 2: All Regions Details ---
            fig, ax = plt.subplots(figsize=(15, 8))
            cmap = matplotlib.colormaps['tab20']
            
            for idx, col in enumerate(cols):
                reg = col.split('_')[-1]
                for slug in REGIONS_SLUGS:
                    if slug in col: reg = slug.replace('_', ' ').title()
                
                if 'Hauts De France' in reg: reg = "Hauts-de-France"
                elif 'Ile De France' in reg: reg = "IDF"
                elif 'Paca' in reg or 'Provence' in reg: reg = "PACA"
                elif 'Bourgogne' in reg: reg = "BFC"
                
                c = cmap(idx % 20)
                if mode == 'long_term':
                    s = df_sub[col].rolling(window_days*24, center=True).mean()
                    ax.plot(s.index, s, color=c, lw=2, label=reg, alpha=0.8)
                else:
                    ax.plot(df_sub.index, df_sub[col], color=c, lw=1.5, label=reg, alpha=0.9)

            if nat_col:
                ax2 = ax.twinx()
                target_ax = ax if 'meteo' in cols[0] else ax2
                
                if mode == 'long_term':
                    s = df_sub[nat_col].rolling(window_days*24, center=True).mean()
                    target_ax.plot(s.index, s, color='black', lw=3, ls='--', label='FRANCE')
                else:
                    target_ax.plot(df_sub.index, df_sub[nat_col], color='black', lw=2.5, ls='--', label='FRANCE')
                
                if target_ax == ax2: ax2.legend(loc='upper right'); ax2.grid(False)

            ax.set_title(f"Graph #{i} : {item['title']} (Regional Breakdown) - {title_pref}", fontsize=14)
            ax.grid(True, alpha=0.3)
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
            
            apply_smart_xaxis(ax)
            plt.tight_layout(); plt.show()
            continue

        # --- Type 5: Simple Pair Comparison (Double Axis if needed) ---
        c1, c2 = item.get("col1"), item.get("col2")
        if not c1 or c1 not in df_sub.columns or c2 not in df_sub.columns: continue
        audit_data.append({"Graph #": i, "Type": "Other", "Variable": c1})
        audit_data.append({"Graph #": i, "Type": "Other", "Variable": c2})

        m1, m2 = df_sub[c1].mean(), df_sub[c2].mean()
        # Use same axis if values are within same order of magnitude (ratio between 0.2 and 5)
        same_axis = 0.2 < abs(m2 / (m1 if m1!=0 else 1)) < 5.0

        fig, ax1 = plt.subplots(figsize=(15, 5))
        ax2 = ax1 if same_axis else ax1.twinx()
        color1 = 'tab:red'
        color2 = 'tab:blue'
        
        props = [(ax1, c1, color1), (ax2, c2, color2)]
        for ax, col, color in props:
            if mode == 'long_term':
                ax.plot(df_sub.index, df_sub[col], color=color, alpha=0.15)
                smooth = df_sub[col].rolling(window_days*24, center=True).mean()
                ax.plot(smooth.index, smooth, color=color, lw=2.5, label=col)
            else:
                ax.plot(df_sub.index, df_sub[col], '.-', color=color, label=col)

            if not same_axis:
                    ax.set_ylabel(col, color=color, fontweight='bold')
                    ax.tick_params(axis='y', labelcolor=color)
                    
        ax1.set_title(f"Graph #{i} : {item.get('title', f'{c1} vs {c2}')}")
        ax1.grid(True, alpha=0.3)
        
        lines, labels = [], []
        for ax in [ax1, ax2]:
            l, lb = ax.get_legend_handles_labels()
            lines.extend(l); labels.extend(lb)
        
        by_label = dict(zip(labels, lines))
        ax1.legend(by_label.values(), by_label.keys(), loc='upper left')
        
        apply_smart_xaxis(ax1)
        plt.show()
        
    return pd.DataFrame(audit_data)

## 4. Interface & Auto-Detect
Define expert comparison pairs and generate the visualization configuration.

In [5]:
EXPERT_COUPLES = [
    # Solar & Radiation
    {"col1": "meteo_radiation_poa_pvgis_wm2_france_solar", "col2": "rte_solar_france", "title": "Radiation: PVGIS POA vs RTE Solar Gen"},
    {"col1": "meteo_radiation_poa_pvgis_wm2_france_solar", "col2": "enedis_production_solar_total_france", "title": "Radiation: PVGIS POA vs Enedis Solar Gen"},
    {"col1": "meteo_radiation_poa_pvgis_wm2_france_solar", "col2": "enedis_meteo_radiation_pseudo_france", "title": "Radiation: PVGIS POA vs Enedis Pseudo-Radiation"},
    {"col1": "meteo_radiation_poa_pvgis_wm2_france_solar", "col2": "meteo_radiation_ghi_era5_wm2_france_solar", "title": "Radiation: PVGIS POA vs ERA5 GHI"},
    {"col1": "meteo_radiation_poa_pvgis_wm2_france_solar", "col2": "meteo_nebulosity_france_solar", "title": "Radiation: PVGIS POA vs Nebulosity"},
    {"col1": "meteo_radiation_poa_pvgis_wm2_france_solar", "col2": "meteo_temperature_celsius_france_solar", "title": "Radiation: PVGIS POA vs Site Temperature"},
    {"col1": "enedis_meteo_radiation_pseudo_france", "col2": "rte_solar_france", "title": "Radiation: Pseudo-Radiation vs RTE Solar Gen"},
    {"col1": "meteo_nebulosity_france_solar", "col2": "rte_solar_france", "title": "Solar: Nebulosity vs RTE Solar Gen"},
    {"col1": "rte_solar_france", "col2": "enedis_production_solar_total_france", "title": "Solar Gen: RTE (TSO) vs Enedis (DSO)"},
    {"col1": "enedis_production_solar_profiled_france", "col2": "enedis_production_solar_total_france", "title": "Solar: Enedis Profiled vs Total"},

    # Wind
    {"col1": "meteo_wind_speed_ms_france_wind", "col2": "rte_wind_france", "title": "Wind: Speed vs RTE Wind Gen"},
    {"col1": "meteo_wind_speed_ms_france_wind", "col2": "enedis_production_wind_total_france", "title": "Wind: Speed vs Enedis Wind Gen"},
    {"col1": "rte_wind_france", "col2": "enedis_production_wind_total_france", "title": "Wind Gen: RTE (TSO) vs Enedis (DSO)"},
    {"col1": "rte_wind_france", "col2": "rte_wind_onshore_france", "title": "Wind Gen: Total vs Onshore (RTE)"},
    {"col1": "meteo_wind_speed_ms_france_wind", "col2": "meteo_air_density_kgm3_france_wind", "title": "Wind: Speed vs Air Density"},
    {"col1": "meteo_wind_speed_ms_france_wind", "col2": "meteo_wind_gust_10min_ms_france_wind", "title": "Wind: Mean Speed vs Gusts"},
    {"col1": "meteo_wind_speed_ms_france_wind", "col2": "meteo_wind_direction_deg_france_wind", "title": "Wind: Speed vs Direction"},
    {"col1": "meteo_air_density_kgm3_france_wind", "col2": "meteo_wind_gust_10min_ms_france_wind", "title": "Wind: Air Density vs Gusts"},
    {"col1": "meteo_wind_speed_ms_france_load", "col2": "meteo_wind_speed_ms_france_wind", "title": "Wind Speed: Load vs Wind Weighted"},
    {"col1": "meteo_pressure_msl_hpa_france_load", "col2": "meteo_wind_speed_ms_france_wind", "title": "Wind: Pressure vs Wind Speed"},

    # Load & Forecasts
    {"col1": "rte_load_france", "col2": "meteo_temperature_celsius_france_load", "title": "Load: RTE vs Temperature (Thermosensitivity)"},
    {"col1": "rte_load_france", "col2": "enedis_load_total_france", "title": "Load: RTE (TSO) vs Enedis (DSO)"},
    {"col1": "rte_load_france", "col2": "meteo_nebulosity_france_load", "title": "Load: RTE vs Nebulosity"},
    {"col1": "rte_load_france", "col2": "rte_solar_france", "title": "Load: RTE vs Solar (Net Load)"},
    {"col1": "rte_load_france", "col2": "rte_forecast_j1_france", "title": "Load: Actual vs Day-Ahead Forecast (J-1)"},
    {"col1": "rte_load_france", "col2": "rte_forecast_j_france", "title": "Load: Actual vs Intraday Forecast (J)"},
    {"col1": "rte_forecast_j_france", "col2": "rte_forecast_j1_france", "title": "Forecast: Intraday (J) vs Day-Ahead (J-1)"},
    {"col1": "enedis_load_residential_total_france", "col2": "enedis_load_professional_total_france", "title": "Load: Residential vs Professional"},
    {"col1": "enedis_load_residential_total_france", "col2": "enedis_load_entreprise_total_france", "title": "Load: Residential vs Enterprise"},
    {"col1": "enedis_load_industrial_total_france", "col2": "enedis_load_total_france", "title": "Load: Industrial (HTA) vs Total"},
    {"col1": "enedis_grid_losses_france", "col2": "enedis_load_total_france", "title": "Load: Grid Losses vs Total"},
    {"col1": "enedis_load_residential_telemetered_france", "col2": "enedis_load_residential_profiled_france", "title": "Residential: Telemetered vs Profiled"},
    {"col1": "enedis_load_professional_telemetered_france", "col2": "enedis_load_professional_profiled_france", "title": "Professional: Telemetered vs Profiled"},
    {"col1": "enedis_load_entreprise_telemetered_france", "col2": "enedis_load_entreprise_profiled_france", "title": "Enterprise: Telemetered vs Profiled"},
    {"col1": "enedis_load_industrial_telemetered_france", "col2": "enedis_load_industrial_profiled_france", "title": "Industrial: Telemetered vs Profiled"},

    # Data Maturity
    {"col1": "rte_load_france", "col2": "rte_load_france_definitive", "title": "Maturity: Best vs Definitive"},
    {"col1": "rte_load_france", "col2": "rte_load_france_consolidated", "title": "Maturity: Best vs Consolidated"},
    {"col1": "rte_load_france", "col2": "rte_load_france_realtime", "title": "Maturity: Best vs Real-Time"},
    {"col1": "rte_load_france_definitive", "col2": "rte_load_france_realtime", "title": "Maturity: Definitive vs Real-Time (Deviation)"},

    # Mix & Exchange
    {"col1": "rte_solar_france", "col2": "rte_wind_france", "title": "Mix: Solar vs Wind (VRE Complementarity)"},
    {"col1": "rte_nuclear_france", "col2": "rte_load_france", "title": "Mix: Nuclear vs Load (Baseload Following)"},
    {"col1": "rte_nuclear_france", "col2": "rte_coal_france", "title": "Mix: Nuclear vs Coal"},
    {"col1": "rte_pump_france", "col2": "rte_hydro_france", "title": "Mix: Pumping vs Hydro"},
    {"col1": "rte_hydro_lake_france", "col2": "rte_hydro_river_france", "title": "Mix: Hydro Lake vs Run-of-River"},
    {"col1": "rte_gas_france", "col2": "rte_bioenergy_france", "title": "Mix: Gas vs Bioenergy"},
    {"col1": "rte_gas_ccg_france", "col2": "rte_gas_tac_france", "title": "Mix: Gas CCGT vs TAC"},
    {"col1": "rte_gas_france", "col2": "rte_co2_rate_france", "title": "Mix: Gas vs CO2 Rate"},
    {"col1": "rte_fuel_france", "col2": "rte_coal_france", "title": "Mix: Fuel vs Coal"},
    {"col1": "rte_hydro_france", "col2": "meteo_precip_3h_mm_france_load", "title": "Mix: Hydro vs Precipitation"},
    {"col1": "rte_exchange_net_france", "col2": "enedis_net_export_to_rte_france", "title": "Exchange: RTE Net vs Enedis TSO Flow"},
    {"col1": "rte_exchange_net_france", "col2": "rte_load_france", "title": "Exchange: Net vs Load"},
    {"col1": "enedis_production_total_france", "col2": "enedis_net_export_to_rte_france", "title": "Enedis: Production vs TSO Export"},

    # Extended Meteo
    {"col1": "meteo_temperature_celsius_france_load", "col2": "meteo_temperature_celsius_france_solar", "title": "Temp: Load Weight vs Solar Weight"},
    {"col1": "meteo_temperature_celsius_france_load", "col2": "enedis_meteo_temperature_real_c_france", "title": "Temp: Meteo-France vs Enedis"},
    {"col1": "enedis_meteo_temperature_real_c_france", "col2": "enedis_meteo_temperature_ref_c_france", "title": "Temp: Enedis Real vs Normal"},
    {"col1": "meteo_temperature_celsius_france_load", "col2": "meteo_dew_point_celsius_france_load", "title": "Temp vs Dew Point"},
    {"col1": "meteo_humidity_percent_france_load", "col2": "meteo_dew_point_celsius_france_load", "title": "Humidity vs Dew Point"},
    {"col1": "meteo_nebulosity_france_load", "col2": "meteo_nebulosity_france_solar", "title": "Nebulosity: Load Weight vs Solar Weight"},
    {"col1": "meteo_precip_1h_mm_france_load", "col2": "meteo_humidity_percent_france_load", "title": "Precipitation vs Humidity"},
    {"col1": "meteo_precip_1h_mm_france_load", "col2": "meteo_precip_3h_mm_france_load", "title": "Precipitation: 1h vs 3h"},
    {"col1": "meteo_pressure_msl_hpa_france_load", "col2": "meteo_pressure_station_hpa_france_load", "title": "Pressure: Sea Level vs Station"},
    {"col1": "meteo_pressure_msl_hpa_france_load", "col2": "meteo_temperature_celsius_france_load", "title": "Pressure vs Temperature"},

    # Calendar
    {"col1": "toy", "col2": "tod", "title": "Calendar: TOY vs TOD"},
    {"col1": "day_type_jf", "col2": "day_type_ljf", "title": "Calendar: Bank Holiday vs Bridge Day"},
    {"col1": "period_summer", "col2": "period_christmas", "title": "Calendar: Summer vs Christmas"},
]

In [ ]:
def generate_hybrid_config(df):
    """
    Generates a visualization configuration JSON by combining:
    1. Known Calendar columns
    2. Physical Balance Checks
    3. Regional Groups
    4. Expert couples (hardcoded above)
    5. Auto-detected Orphan columns (paired with Load to ensure they are plotted)
    """
    all_cols = sorted(df.select_dtypes(include=np.number).columns)
    final_config = []
    covered_cols = set()

    # 1. Calendar
    cal_cols = get_calendar_group(all_cols)
    cal_themes = {
        "Vacation Zones": ['period_holiday_zone'],
        "Day Types": ['day_type_'],
        "Seasons & Periods": ['period_summer', 'period_christmas', 'period_holiday', 'period_hour_changed'],
        "Technical": ['toy', 'tod', 'month','year','week_number','validity'],
    }
    for theme, tags in cal_themes.items():
        cols = [c for c in cal_cols if any(t in c.lower() for t in tags) and c not in covered_cols]
        if cols:
            final_config.append({"type": "calendar_lines", "cols": sorted(cols), "title": f"Calendar: {theme}"})
            covered_cols.update(cols)

    # 2. Balance & Regions
    # A. Physical Balance
    balance_checks = get_balance_check(all_cols)
    if balance_checks:
        final_config.extend(balance_checks)
        for b in balance_checks:
            covered_cols.update(b['In'])
            covered_cols.update(b['Out'])
            
    # B. Regional Groups
    regional_groups = get_regional_groups(all_cols)
    if regional_groups:
        final_config.extend(regional_groups)
        for g in regional_groups:
            covered_cols.update(g['cols'])

    # 3. Expert List
    for item in EXPERT_COUPLES:
        c1, c2 = item.get('col1'), item.get('col2')
        if c1 in df.columns and c2 in df.columns:
            if c1 in covered_cols and c2 in covered_cols: continue
            final_config.append(item)
            covered_cols.add(c1); covered_cols.add(c2)

    # 4. Logical Orphans
    remaining = [c for c in all_cols if c not in covered_cols]
    orphan_patterns = {
        'RTE Forecast': ['rte_forecast_'],
        'Thermal/Other': ['rte_fuel_', 'rte_bio_', 'rte_gas_'],
        'Hydro Detail': ['rte_hydro_lake', 'rte_hydro_river'], 
        'Enedis Detail': ['enedis_load_industrial', 'enedis_production_', 'enedis_meteo_'],
        'Meteo Detail': ['meteo_dew_point', 'meteo_precip', 'meteo_pressure']
    }
    
    for theme, tags in orphan_patterns.items():
        matches = [c for c in remaining if any(t in c for t in tags) and c not in covered_cols]
        for i in range(0, len(matches) - 1, 2):
            c1, c2 = matches[i], matches[i+1]
            final_config.append({"col1": c1, "col2": c2, "title": f"Auto [{theme}]: {c1} vs {c2}"})
            covered_cols.update([c1, c2])
        if len(matches) % 2 != 0:
            orphan = matches[-1]
            if orphan not in covered_cols:
                final_config.append({"col1": orphan, "col2": "enedis_load_total_france", "title": f"Auto [{theme}]: {orphan}"})
                covered_cols.add(orphan)

    # 5. True Orphans
    for c in [c for c in all_cols if c not in covered_cols]:
        final_config.append({"col1": c, "col2": "enedis_load_total_france", "title": f"ORPHAN: {c}"})

    return final_config

# --- Widgets ---
txt_conf = widgets.Textarea(placeholder="JSON Config...", layout={'width': '98%', 'height': '300px'})
w_dates = [widgets.DatePicker(description='Start', value=pd.to_datetime('2024-01-01')), 
           widgets.DatePicker(description='End', value=pd.to_datetime('2024-12-31'))]
w_win = widgets.IntSlider(value=7, min=1, max=30, description='Win (d)')
btn_gen = widgets.Button(description='⚡ Generate Config', button_style='warning')
btn_run = widgets.Button(description='🚀 Run Audit', button_style='success')
btn_mode = widgets.ToggleButtons(options=['Long Term', 'Cold Zoom'], description='Mode')
out_final = widgets.Output()

def on_gen(b):
    if df_main is not None:
        cfg = generate_hybrid_config(df_main)
        txt_conf.value = str(cfg).replace("},", "},\n")
    else: print("⚠️ Load data first (Cell 2)")

def on_run(b):
    with out_final:
        clear_output()
        if df_main is None: return print("⚠️ Load data first")
        try:
            cfg = ast.literal_eval(txt_conf.value)
            audit_df = plot_analysis(df_main, cfg, str(w_dates[0].value), str(w_dates[1].value), 
                                     'long_term' if btn_mode.value=='Long Term' else 'zoom_cold', w_win.value)
            if audit_df is not None:
                print("\n" + "="*50 + "\n📋 COVERAGE TABLE\n" + "="*50)
                orphans = audit_df[audit_df['Type'].astype(str).str.contains("Orphan", case=False, na=False)]
                if not orphans.empty:
                    print("🚨 WARNING: Unclassified columns detected:")
                    display(orphans)
                else: print("✅ SUCCESS: 100% of columns are classified.")
        except Exception as e: print(f"❌ Error: {e}")

btn_gen.on_click(on_gen)
btn_run.on_click(on_run)

display(widgets.VBox([
    widgets.HTML("<h3>🔍 Audit & Visualization (Final Version)</h3>"),
    btn_gen, txt_conf,
    widgets.HBox(w_dates), widgets.HBox([w_win, btn_mode]),
    btn_run, out_final
]))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
import numpy as np

def generate_enhanced_validation_figure(df, start_date="2024-01-09", end_date="2024-01-19"):
    """
    Generates a publication-ready validation plot with difference shading and statistics.
    """
    if df is None:
        print("❌ Error: No dataset loaded.")
        return

    # 1. Prepare Data
    mask = (df.index >= start_date) & (df.index <= end_date)
    plot_data = df.loc[mask].copy()
    
    if plot_data.empty:
        print("⚠️ No data in range.")
        return

    # 2. Setup Style
    sns.set_theme(style="ticks", context="paper", font_scale=1.4)
    # Using a high-contrast palette safe for colorblindness/printing
    COLORS = {"rte": "#D55E00", "enedis": "#0072B2", "fill_a": "#e6f2ff", "fill_b": "#ffe6cc"}
    
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 10), sharex=True)

    # === PANEL A: Physics-Awareness ===
    col_solar = "meteo_temperature_celsius_france_solar"
    col_load = "meteo_temperature_celsius_france_load"
    
    if all(c in plot_data.columns for c in [col_solar, col_load]):
        # Calculate Stats
        diff = (plot_data[col_solar] - plot_data[col_load]).mean()
        corr = plot_data[col_solar].corr(plot_data[col_load])
        
        # Plot Lines
        ax1.plot(plot_data.index, plot_data[col_solar], color=COLORS["enedis"], label=f"Solar-Weighted (Generation)", lw=2)
        ax1.plot(plot_data.index, plot_data[col_load], color=COLORS["rte"], label=f"Load-Weighted (Population)", lw=2, linestyle="--")
        
        # Shading the Difference
        ax1.fill_between(plot_data.index, plot_data[col_load], plot_data[col_solar], 
                         color=COLORS["enedis"], alpha=0.15, label="Spatial Divergence")

        ax1.set_ylabel("Temperature (°C)", fontweight="bold")
        ax1.set_title(f"A. Physics-Awareness: Impact of Spatial Weighting (Avg Δ = {diff:.2f}°C)", loc="left", fontweight="bold")
        ax1.legend(loc="upper left", frameon=True, framealpha=0.95)
        ax1.grid(True, alpha=0.3)

    # === PANEL B: Harmonization ===
    col_rte = "rte_load_france"
    col_enedis = "enedis_load_total_france"
    
    if all(c in plot_data.columns for c in [col_rte, col_enedis]):
        # Plot Lines
        ax2.plot(plot_data.index, plot_data[col_rte], color=COLORS["rte"], label="National Load (RTE / TSO)", lw=2)
        ax2.plot(plot_data.index, plot_data[col_enedis], color=COLORS["enedis"], label="Distribution Load (Enedis / DSO)", lw=2)
        
        # Shading the Gap (Industrial + Losses)
        ax2.fill_between(plot_data.index, plot_data[col_enedis], plot_data[col_rte], 
                         color="gray", alpha=0.1, hatch='///', label="Gap: Industry & Losses")

        # Refined Annotation
        mid_idx = len(plot_data) // 2
        mid_date = plot_data.index[mid_idx]
        val_rte = plot_data.iloc[mid_idx][col_rte]
        val_enedis = plot_data.iloc[mid_idx][col_enedis]
        
        ax2.annotate(
            f'Transmission Gap\n~{(val_rte - val_enedis)/1000:.1f} GW',
            xy=(mid_date, (val_rte + val_enedis)/2),
            xytext=(mid_date + pd.Timedelta(hours=18), val_rte),
            arrowprops=dict(facecolor='black', arrowstyle='->', connectionstyle="arc3,rad=-0.2"),
            fontsize=11, backgroundcolor="white", ha='left'
        )

        ax2.set_ylabel("Power (MW)", fontweight="bold")
        ax2.set_title("B. Data Harmonization: TSO vs. DSO Alignment", loc="left", fontweight="bold")
        ax2.legend(loc="upper right", frameon=True, framealpha=0.95)
        ax2.grid(True, alpha=0.3)

    # 3. Formatting
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax2.set_xlabel("Date (2024)", fontweight="bold")
    
    plt.tight_layout()
    
    # Save
    save_path = "paper/images/validation_stacked.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Enhanced Figure generated: {save_path}")
    plt.show()

# Run
if 'df_main' in globals() and df_main is not None:
    generate_enhanced_validation_figure(df_main)
else:
    print("⚠️ Please load a dataset first.")

In [8]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
import numpy as np

def generate_maturity_forensics_figure(df):
    """
    Generates a forensic analysis of Data Maturity (Real-Time vs Definitive).
    Automatically finds the period of maximum deviation to illustrate the value of the 'Golden Record'.
    """
    if df is None:
        print("❌ Error: No dataset loaded.")
        return

    # 1. Identify Columns
    col_rt = "rte_load_france_realtime"
    col_def = "rte_load_france_definitive"
    
    if not all(c in df.columns for c in [col_rt, col_def]):
        print(f"⚠️ Missing columns: {col_rt} or {col_def}")
        return

    # 2. Heuristic: Find interesting window (Max Deviation)
    # We look for a recent period (e.g., 2023-2024) where both exist
    # Note: 'Definitive' might lag by 12 months. We use 'Consolidated' as proxy if Definitive is missing.
    # For safety, let's filter for where both are non-null.
    valid_data = df.dropna(subset=[col_rt, col_def]).copy()
    
    if valid_data.empty:
        print("⚠️ No overlapping data found between Real-Time and Definitive streams.")
        return

    valid_data["delta_abs"] = (valid_data[col_def] - valid_data[col_rt]).abs()
    
    # Find the date of max error
    worst_moment = valid_data["delta_abs"].idxmax()
    
    # Define a 5-day window around the worst moment
    start_date = worst_moment - pd.Timedelta(days=2)
    end_date = worst_moment + pd.Timedelta(days=3)
    
    plot_data = valid_data.loc[start_date:end_date]
    
    # 3. Setup Plot
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.4)
    COLORS = {"rt": "#e74c3c", "def": "#2ecc71", "delta": "#34495e"}
    
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 10), sharex=True, 
                                   gridspec_kw={'height_ratios': [3, 1]})

    # === PANEL A: The Signal Divergence ===
    ax1.plot(plot_data.index, plot_data[col_rt], color=COLORS["rt"], label="Real-Time Stream (Provisional)", lw=1.5, alpha=0.7, linestyle="--")
    ax1.plot(plot_data.index, plot_data[col_def], color=COLORS["def"], label="Definitive Stream (Validated)", lw=2.5)
    
    # Highlight the specific zone of error
    ax1.fill_between(plot_data.index, plot_data[col_rt], plot_data[col_def], color=COLORS["rt"], alpha=0.1)

    ax1.set_ylabel("Power Load (MW)", fontweight="bold")
    ax1.set_title(f"Data Maturity Forensics: The 'Golden Record' Correction", loc="left", fontweight="bold")
    ax1.legend(loc="upper right", frameon=True, framealpha=0.95)
    
    # === PANEL B: The Quantitative Error ===
    delta = plot_data[col_def] - plot_data[col_rt]
    ax2.plot(delta.index, delta, color=COLORS["delta"], lw=1.5, label="Correction Delta (Definitive - RT)")
    
    # Fill positive/negative errors
    ax2.fill_between(delta.index, delta, 0, where=(delta > 0), color=COLORS["def"], alpha=0.3)
    ax2.fill_between(delta.index, delta, 0, where=(delta < 0), color=COLORS["rt"], alpha=0.3)
    
    ax2.axhline(0, color="black", lw=1)
    
    # Stats Annotation
    max_err = delta.abs().max()
    rmse = np.sqrt((delta**2).mean())
    ax2.text(0.02, 0.9, f"Max Error: {max_err:.0f} MW\nRMSE: {rmse:.0f} MW", 
             transform=ax2.transAxes, bbox=dict(facecolor='white', alpha=0.8))

    ax2.set_ylabel("Correction (MW)", fontweight="bold")
    ax2.legend(loc="lower right")

    # 4. Formatting
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
    ax2.set_xlabel(f"Date ({worst_moment.year})", fontweight="bold")
    
    plt.tight_layout()
    
    # Output
    save_path = "paper/images/maturity_forensics.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Maturity Figure generated: {save_path} (Focus: {worst_moment.date()})")
    plt.show()

# Run
if 'df_main' in globals() and df_main is not None:
    generate_maturity_forensics_figure(df_main)

In [9]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
import numpy as np

def generate_spatial_breakdown_figure(df, tech="wind", start_date="2024-01-09", end_date="2024-01-16"):
    """
    Generates a Stacked Area Chart showing regional contributions to a national total.
    Validates the spatial aggregation logic and data completeness.
    """
    if df is None:
        print("❌ Error: No dataset loaded.")
        return

    # 1. Prepare Data Window
    mask = (df.index >= start_date) & (df.index <= end_date)
    plot_data = df.loc[mask].copy()
    
    if plot_data.empty:
        print("⚠️ No data in range.")
        return

    # 2. Dynamic Column Identification
    prefix = f"rte_{tech}_"
    national_col = f"rte_{tech}_france"
    
    # Check national column existence
    if national_col not in plot_data.columns:
        print(f"⚠️ Missing national column: {national_col}")
        return

    # ROBUST FILTERING LOGIC
    regional_cols = []
    for c in plot_data.columns:
        if not c.startswith(prefix): continue
        if c == national_col: continue          # Exclude the National Aggregate itself
        if "onshore" in c or "offshore" in c: continue  # Exclude National Subtypes

        # "False Friend" Check:
        # We want to exclude "rte_wind_france" (already done above)
        # But KEEP "rte_wind_ile_de_france" and "rte_wind_hauts_de_france"
        if "france" in c:
            if not ("ile_de_france" in c or "hauts_de_france" in c):
                continue 
        
        regional_cols.append(c)
    
    if not regional_cols:
        print(f"⚠️ No regional columns found for {tech}.")
        return

    # 3. Sort Regions by Volume (Largest at bottom for stability)
    sums = plot_data[regional_cols].sum().sort_values(ascending=False)
    sorted_regions = sums.index.tolist()
    
    # 4. Plotting
    sns.set_theme(style="white", context="paper", font_scale=1.4)
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Clean Labels for Legend (Remove prefix and underscores)
    labels = [c.replace(prefix, "").replace("_", " ").title().replace("De", "de") for c in sorted_regions]
    
    # Stackplot
    ax.stackplot(plot_data.index, plot_data[sorted_regions].T, labels=labels, alpha=0.85, edgecolor='white', linewidth=0.5)
    
    # Overlay National Signal (The "Check")
    ax.plot(plot_data.index, plot_data[national_col], color="black", lw=2.5, linestyle="--", label="National Total (RTE Source)")

    # 5. Styling
    ax.set_ylabel(f"{tech.title()} Generation (MW)", fontweight="bold")
    ax.set_title(f"Spatial Deconvolution: Regional {tech.title()} Contribution to National Grid", loc="left", fontweight="bold", fontsize=16)
    
    # Legend - Outside to save space
    handles, lbls = ax.get_legend_handles_labels()
    # Reverse legend to match stack order (visual correspondence)
    ax.legend(reversed(handles), reversed(lbls), loc='upper left', bbox_to_anchor=(1.01, 1), title="Regions (Sorted by Volume)")
    
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_xlim(plot_data.index.min(), plot_data.index.max())
    
    # Date Formatting
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d\n%H:%M'))
    ax.set_xlabel(f"Date ({pd.to_datetime(start_date).year})", fontweight="bold")

    plt.tight_layout()
    
    # Output
    save_path = f"paper/images/spatial_breakdown_{tech}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Spatial Figure generated: {save_path}")
    plt.show()

# Run
if 'df_main' in globals() and df_main is not None:
    generate_spatial_breakdown_figure(df_main, tech="solar")
    generate_spatial_breakdown_figure(df_main, tech="wind")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
import numpy as np
import os

def _plot_dashboard(data, title_prefix, filename, output_dir):
    """
    Helper function: Generates, saves, and displays a standardized observability dashboard.
    
    Parameters
    ----------
    data : pd.DataFrame
        DataFrame containing exactly two columns: ['Telemetered', 'Profiled'].
    title_prefix : str
        String to prefix the chart titles (e.g., "Total", "Residential").
    filename : str
        Filename for the saved image.
    output_dir : str
        Directory to save the image.
    """
    # Resample to daily mean and calculate coverage rate
    daily_data = data.resample('D').mean()
    total_load = daily_data['Telemetered'] + daily_data['Profiled']
    
    daily_data['Coverage_Rate'] = np.where(total_load > 0, 
                                           (daily_data['Telemetered'] / total_load) * 100, 
                                           0)

    # Compute average diurnal cycle (00:00 - 23:30)
    avg_profile = data.groupby(data.index.time).mean()
    dummy_index = pd.date_range("2024-01-01", periods=len(avg_profile), freq="30min")
    avg_profile.index = dummy_index

    # --- Plotting ---
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.4)
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(14, 12), constrained_layout=True)

    # Panel A: Long-term Trend (Stacked Area)
    colors = ["#bdc3c7", "#2ecc71"]
    labels = ["Profiled (Estimated)", "Telemetered (Real)"]
    
    ax1.stackplot(daily_data.index, 
                  daily_data['Profiled'], 
                  daily_data['Telemetered'],
                  labels=labels, colors=colors, alpha=0.9)
    
    # Twin Axis for Coverage Rate
    ax1_twin = ax1.twinx()
    ax1_twin.plot(daily_data.index, daily_data['Coverage_Rate'], 
                  color="#27ae60", lw=2.5, ls="--", label="Telemetry Rate (%)")
    
    ax1.set_title(f"A. {title_prefix}: Rising Observability Trend", loc="left", fontweight="bold")
    ax1.set_ylabel("Daily Mean Load (MW)", fontweight="bold")
    ax1_twin.set_ylabel("Smart Meter Coverage (%)", fontweight="bold", color="#27ae60")
    ax1_twin.tick_params(axis='y', labelcolor="#27ae60")
    ax1_twin.set_ylim(0, 105)
    ax1_twin.grid(False)
    
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1_twin.get_legend_handles_labels()
    ax1.legend(h1+h2, l1+l2, loc='upper left', frameon=True, framealpha=0.95)
    
    # Date Formatting: Month Year (e.g., "Jun 2021")
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax1.set_xlim(daily_data.index.min(), daily_data.index.max())

    # Panel B: Average Daily Profile (Dual Axis)
    ax2.plot(avg_profile.index, avg_profile['Telemetered'], color="#2ecc71", lw=3, label="Telemetered")
    ax2.set_ylabel("Telemetered Power (MW)", fontweight="bold", color="#2ecc71")
    ax2.tick_params(axis='y', labelcolor="#2ecc71")
    
    ax2_twin = ax2.twinx()
    ax2_twin.plot(avg_profile.index, avg_profile['Profiled'], color="#95a5a6", lw=3, ls="--", label="Profiled")
    ax2_twin.set_ylabel("Profiled Power (MW)", fontweight="bold", color="#7f8c8d")
    ax2_twin.tick_params(axis='y', labelcolor="#7f8c8d")
    ax2_twin.grid(False)

    ax2.set_title(f"B. {title_prefix}: Average Diurnal Cycle (Shape Correlation)", loc="left", fontweight="bold")
    ax2.set_xlabel("Time of Day", fontweight="bold")
    
    h3, l3 = ax2.get_legend_handles_labels()
    h4, l4 = ax2_twin.get_legend_handles_labels()
    ax2.legend(h3+h4, l3+l4, loc='upper center', frameon=True, framealpha=0.95, ncol=2)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax2.set_xlim(dummy_index.min(), dummy_index.max())
    
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show() 
    print(f"✅ Generated: {save_path}")


def generate_observability_dashboards(df, start_date="2020-01-01", end_date="2026-01-01"):
    """
    Orchestrator function:
    1. Prepares data (filters by date).
    2. Generates the 'Total' aggregated dashboard.
    3. Generates individual dashboards for 'residential', 'professional', 'entreprise'.
    """
    if df is None:
        print("❌ Error: No dataset loaded.")
        return

    mask = (df.index >= start_date) & (df.index <= end_date)
    plot_data_global = df.loc[mask].copy()
    
    if plot_data_global.empty:
        print("⚠️ No data in range.")
        return

    output_dir = "paper/images"
    os.makedirs(output_dir, exist_ok=True)
    sectors = ['residential', 'professional', 'entreprise']

    # --- Generate TOTAL Dashboard ---
    print("🔄 Processing: Total Aggregation...")
    
    total_tel = pd.Series(0, index=plot_data_global.index)
    total_prof = pd.Series(0, index=plot_data_global.index)
    
    valid_cols = False
    for s in sectors:
        col_tel = f"enedis_load_{s}_telemetered_france"
        col_prof = f"enedis_load_{s}_profiled_france"
        
        if col_tel in plot_data_global.columns and col_prof in plot_data_global.columns:
            total_tel += plot_data_global[col_tel]
            total_prof += plot_data_global[col_prof]
            valid_cols = True
    
    if valid_cols:
        df_total = pd.DataFrame({'Telemetered': total_tel, 'Profiled': total_prof})
        _plot_dashboard(df_total, "Total Grid", "enedis_observability_composite.png", output_dir)
    else:
        print("⚠️ Could not generate Total graph: Missing columns.")

    # --- Generate SECTOR Dashboards ---
    for s in sectors:
        print(f"🔄 Processing: {s}...")
        col_tel = f"enedis_load_{s}_telemetered_france"
        col_prof = f"enedis_load_{s}_profiled_france"
        
        if col_tel in plot_data_global.columns and col_prof in plot_data_global.columns:
            df_sector = plot_data_global[[col_tel, col_prof]].rename(columns={
                col_tel: 'Telemetered',
                col_prof: 'Profiled'
            })
            _plot_dashboard(df_sector, s.capitalize(), f"enedis_observability_{s}.png", output_dir)
        else:
            print(f"⚠️ Skipping {s}: Missing columns.")

# Run
if 'df_main' in globals() and df_main is not None:
    generate_observability_dashboards(df_main)
else:
    print("⚠️ Please load a dataset (df_main) using the widget above.")

In [ ]:
w_col_select = widgets.Dropdown(description='Column:', layout={'width': '400px'})
w_start_cmp = widgets.DatePicker(description='Start', value=pd.to_datetime('2024-01-01'))
w_end_cmp = widgets.DatePicker(description='End', value=pd.to_datetime('2024-01-03'))
btn_refresh_cols = widgets.Button(description='🔄 Refresh Cols', button_style='info')
btn_plot_cmp = widgets.Button(description='📊 Plot Comparison', button_style='primary')
out_cmp = widgets.Output()

def update_common_columns(b):
    """Finds columns that exist in both 15min and 30min datasets."""
    if df_15 is not None and df_30 is not None:
        cols_15 = set(df_15.select_dtypes(include=np.number).columns)
        cols_30 = set(df_30.select_dtypes(include=np.number).columns)
        common = sorted(list(cols_15 & cols_30))
        
        w_col_select.options = common

        if 'enedis_load_total_france' in common:
            w_col_select.value = 'enedis_load_total_france'
            
        print(f"✅ {len(common)} common columns identified.")
    else:
        print("⚠️ Please load both 15min and 30min datasets first (Cell 2).")

def plot_visual_check(b):
    with out_cmp:
        clear_output(wait=True)
        col = w_col_select.value
        
        if df_15 is None or df_30 is None:
            print("⚠️ Error: Missing datasets.")
            return
        if not col:
            print("⚠️ Error: No column selected.")
            return
            
        d_start_str = w_start_cmp.value.strftime('%Y-%m-%d')
        d_end_str = w_end_cmp.value.strftime('%Y-%m-%d')
        
        print(f"⏳ Generating plot for {d_start_str} to {d_end_str}...")

        iso_start, iso_end = str(w_start_cmp.value), str(w_end_cmp.value)
        d15 = df_15.loc[(df_15.index >= iso_start) & (df_15.index <= iso_end)]
        d30 = df_30.loc[(df_30.index >= iso_start) & (df_30.index <= iso_end)]
        
        if d15.empty:
            print(f"⚠️ No data found between {d_start_str} and {d_end_str}.")
            return

        fig, ax = plt.subplots(figsize=(15, 6))
        
        ax.plot(d15.index, d15[col], color="#1b0fc0", marker='.', markersize=5, 
                linewidth=1, alpha=0.6, label='15 min (Source)')
        
        ax.plot(d30.index, d30[col], color="#830b0b", marker='o', markersize=6, 
                linestyle='--', linewidth=2, fillstyle='none', label='30 min (Aggregated)')
        
        ax.set_title(f"Resolution Inspection: {col} ({d_start_str} - {d_end_str})")
        ax.set_ylabel("Value")
        
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
        
        ax.grid(True, alpha=0.3)
        ax.legend()
        
        plt.tight_layout()
        plt.show()

btn_refresh_cols.on_click(update_common_columns)
btn_plot_cmp.on_click(plot_visual_check)

ui_cmp = widgets.VBox([
    widgets.HTML("<h3>🔍 Visual Inspection: 15min vs 30min</h3>"),
    widgets.HBox([btn_refresh_cols, w_col_select]),
    widgets.HBox([w_start_cmp, w_end_cmp, btn_plot_cmp]),
    out_cmp
])

display(ui_cmp)

In [ ]:
def audit_dataset_structure(df, dataset_name):
    """
    Expert Audit: Categorizes columns, differentiates Global vs Technical availability, and counts Zeros.
    """
    if df is None:
        print(f"⚠️ Dataset '{dataset_name}' empty or not loaded.")
        return

    # --- 0. TIME INDEX PREPARATION ---
    # Essential for "Technical Availability" calculation (temporal slicing)
    working_df = df.copy()
    print(list(working_df.columns))
    if 'date' in working_df.columns:
        working_df.set_index('date', inplace=True)
    
    if not isinstance(working_df.index, pd.DatetimeIndex):
        try:
            working_df.index = pd.to_datetime(working_df.index)
        except:
            print("❌ Critical Error: Unable to parse index as datetime.")
            return

    print(f"🔎 EXPERT AUDIT: {dataset_name}")
    print(f"📅 Date Range: {working_df.index.min().date()} to {working_df.index.max().date()}")
    print("=" * 80)

    # --- 1. SORTING LOGIC ---
    def get_category_data(col):
        c = col.lower()
        if 'enedis_' in c: return ('Enedis', 1)
        if 'meteo_' in c: return ('Meteo', 2)
        if 'rte_' in c: return ('RTE', 3)
        return ('Other', 4)

    # --- 2. ADVANCED STATS ---
    stats_data = []
    
    # Total dataset rows (for global availability)
    total_len = len(working_df)
    
    for col in working_df.columns:
        series = working_df[col]
        
        # A. Basic Metrics
        n_missing = series.isna().sum()
        n_zeros = (series == 0).sum()
        is_numeric = pd.api.types.is_numeric_dtype(series)
        
        # B. Temporal Metrics (Real Start / End of Signal)
        first_valid_idx = series.first_valid_index()
        last_valid_idx = series.last_valid_index()
        
        if first_valid_idx is None:
            # Empty column
            tech_avail = 0.0
            start_date, end_date = "-", "-"
        else:
            start_date = first_valid_idx
            end_date = last_valid_idx
            
            # C. TECHNICAL Availability
            # We slice the series between its true start and end
            # And check for gaps INSIDE this period
            sub_series = series.loc[first_valid_idx:last_valid_idx]
            tech_len = len(sub_series)
            tech_missing = sub_series.isna().sum()
            tech_avail = 100 * (1 - tech_missing / tech_len)

        # D. Categorization
        cat_name, cat_score = get_category_data(col)
        
        stats_data.append({
            'Column': col,
            'Group': cat_name,
            '_sort_score': cat_score,
            'Start Date': start_date.date() if isinstance(start_date, pd.Timestamp) else start_date,
            'End Date': end_date.date() if isinstance(end_date, pd.Timestamp) else end_date,
            'Global Avail. (%)': round(100 * (1 - n_missing/total_len), 1),
            'Tech. Avail. (%)': round(tech_avail, 2), # Key metric
            'Zeros (%)': round(100 * (n_zeros/total_len), 2),
            'Min': round(series.min(), 2) if is_numeric else np.nan,
            'Mean': round(series.mean(), 2) if is_numeric else np.nan,
            'Max': round(series.max(), 2) if is_numeric else np.nan
        })

    if not stats_data: return

    # --- 3. DATAFRAME CREATION & SORTING ---
    df_stats = pd.DataFrame(stats_data)
    df_stats = df_stats.sort_values(by=['_sort_score', 'Column'], ascending=[True, True])
    df_stats.set_index('Column', inplace=True)

    # --- 4. TABLE DISPLAY ---
    print("\n📋 DETAILED TABLE")
    print("   > Global Avail.: % presence over the entire file duration.")
    print("   > Tech. Avail.: % presence between the specific Start and End of the signal (detects holes).")
    
    cols_order = ['Group', 'Start Date', 'End Date', 'Global Avail. (%)', 'Tech. Avail. (%)', 'Zeros (%)', 'Min', 'Mean', 'Max']
    
    try:
        # Conditional Styling
        styler = df_stats[cols_order].style\
            .background_gradient(subset=['Global Avail. (%)'], cmap='RdYlGn', vmin=0, vmax=100)\
            .background_gradient(subset=['Tech. Avail. (%)'], cmap='Greens', vmin=90, vmax=100)\
            .background_gradient(subset=['Zeros (%)'], cmap='Reds', vmin=0, vmax=50)\
            .format("{:.2f}", subset=['Min','Mean','Max'])
        display(styler)
    except:
        display(df_stats[cols_order])

    # --- 5. CHRONOLOGICAL HEATMAP ---
    print("\n🧩 CHRONOLOGICAL HEATMAP (Based on Time Index)")
    
    # Use sorted index order for Y-axis
    availability = working_df[df_stats.index].notna().astype(int)
    
    # Weekly Resample for readability
    resampled = availability.resample('W').mean().T
    
    height = max(8, len(df_stats) * 0.35)
    plt.figure(figsize=(20, height))
    
    ax = sns.heatmap(resampled, 
                cmap="RdYlGn", 
                cbar_kws={'label': 'Fill Rate'}, 
                vmin=0, vmax=1, 
                linewidths=0) 
    
    plt.title(f"Temporal Availability: {dataset_name}", fontsize=16)
    plt.xlabel("Years")
    
    # White separation lines between groups
    groups = df_stats['Group']
    change_positions = np.where(groups.ne(groups.shift()).values)[0]
    for pos in change_positions:
        if pos > 0:
            ax.axhline(y=pos, color='white', linewidth=3, linestyle='-')

    plt.tight_layout()
    plt.show()

# --- 2. WIDGETS ---

w_audit_target = widgets.Dropdown(
    options=['15 minutes', '30 minutes', 'Main (Active)'],
    value='Main (Active)',
    description='Target:',
    style={'description_width': 'initial'}
)

btn_run_audit = widgets.Button(
    description='🔍 Run Expert Audit',
    button_style='primary',
    layout={'width': '250px'},
    icon='check-square'
)

out_audit = widgets.Output()

# --- 3. HANDLER ---

def on_audit_click(b):
    with out_audit:
        clear_output(wait=True)
        
        target_df = None
        name = w_audit_target.value
        
        if name == '15 minutes':
            if 'df_15' in globals() and df_15 is not None: target_df = df_15
        elif name == '30 minutes':
            if 'df_30' in globals() and df_30 is not None: target_df = df_30
        elif name == 'Main (Active)':
            if 'df_main' in globals() and df_main is not None: target_df = df_main
            
        if target_df is not None:
            audit_dataset_structure(target_df, f"Dataset {name}")
        else:
            print(f"⚠️ Dataset '{name}' is not loaded.")

btn_run_audit.on_click(on_audit_click)

# --- 4. DISPLAY ---

ui_audit = widgets.VBox([
    widgets.HTML("<h3>🚀 Expert Audit: Technical Availability & Zeros</h3>"),
    widgets.HBox([w_audit_target, btn_run_audit]),
    out_audit
])

display(ui_audit)